# Project 2: Data Understanding and Dataset Verification

## Purpose

Project 1 focused on checking whether the retail orders dataset was clean and reliable.

Project 2 focuses on Exploratory Data Analysis (EDA).  
Before starting EDA, this notebook performs a quick verification to confirm that the dataset is ready for analysis.

Main question:

Can this dataset be safely used to discover patterns, trends, outliers, and business insights?

In [1]:
import pandas as pd
import numpy as np

In [6]:
cleaned_path = "../data/processed/orders_cleaned.csv"
raw_path = "../data/raw/Dataset for Data Analytics.xlsx"
try:
    df = pd.read_csv(cleaned_path)
    print("Loaded cleaned dataset from project 1")
except FileNotFoundError:
    print("Cleaned dataset not found.")
df.head()

Loaded cleaned dataset from project 1


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [7]:
print("Number of rows:", df.shape[0])
print("Number of Columns:", df.shape[1])

Number of rows: 1200
Number of Columns: 14


In [8]:
df.columns

Index(['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'UnitPrice',
       'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber',
       'ItemsInCart', 'CouponCode', 'ReferralSource', 'TotalPrice'],
      dtype='object')

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   OrderID          1200 non-null   object 
 1   Date             1200 non-null   object 
 2   CustomerID       1200 non-null   object 
 3   Product          1200 non-null   object 
 4   Quantity         1200 non-null   int64  
 5   UnitPrice        1200 non-null   float64
 6   ShippingAddress  1200 non-null   object 
 7   PaymentMethod    1200 non-null   object 
 8   OrderStatus      1200 non-null   object 
 9   TrackingNumber   1200 non-null   object 
 10  ItemsInCart      1200 non-null   int64  
 11  CouponCode       1200 non-null   object 
 12  ReferralSource   1200 non-null   object 
 13  TotalPrice       1200 non-null   float64
dtypes: float64(2), int64(2), object(10)
memory usage: 131.4+ KB


In [10]:
# Convert Date column to datetime format if needed

if not pd.api.types.is_datetime64_any_dtype(df["Date"]):
    if pd.api.types.is_numeric_dtype(df["Date"]):
        df["Date"] = pd.to_datetime(df["Date"], origin="1899-12-30", unit="D")
    else:
        df["Date"] = pd.to_datetime(df["Date"])

df["Date"].head()

0   2023-01-04
1   2024-08-23
2   2024-02-27
3   2023-10-15
4   2025-05-08
Name: Date, dtype: datetime64[ns]

In [11]:
missing_values = df.isnull().sum().sort_values(ascending=False)
missing_values

OrderID            0
Date               0
CustomerID         0
Product            0
Quantity           0
UnitPrice          0
ShippingAddress    0
PaymentMethod      0
OrderStatus        0
TrackingNumber     0
ItemsInCart        0
CouponCode         0
ReferralSource     0
TotalPrice         0
dtype: int64

In [19]:
# Create a new column to identify whether a coupon was used

coupon_clean = df["CouponCode"].astype("string").str.strip().str.lower()

no_coupon_values = ["", "nan", "none", "null", "no coupon", "no_coupon", "n/a", "na"]

df["HasCoupon"] = np.where(
    df["CouponCode"].isnull() | coupon_clean.isin(no_coupon_values),
    "No Coupon",
    "Coupon Used"
)

df["HasCoupon"].value_counts()

HasCoupon
Coupon Used    891
No Coupon      309
Name: count, dtype: int64

In [13]:
duplicate_order_ids = df["OrderID"].duplicated().sum()
print("Duplicate Order IDs:", duplicate_order_ids)

Duplicate Order IDs: 0


In [15]:
print("Start date:", df["Date"].min()) #check date range
print("End date:", df["Date"].max())

Start date: 2023-01-01 00:00:00
End date: 2025-06-30 00:00:00


In [16]:
df["CalculatedTotalPrice"] = df["Quantity"] * df["UnitPrice"] #check total price calculation

price_mismatch = np.isclose(
    df["TotalPrice"],
    df["CalculatedTotalPrice"],
    atol=0.01
) == False

print("TotalPrice mismatches:", price_mismatch.sum())

TotalPrice mismatches: 0


In [20]:
summary = {
    "Total Orders": df["OrderID"].nunique(),
    "Total Customers": df["CustomerID"].nunique(),
    "Total Revenue": df["TotalPrice"].sum(),
    "Average Order Value": df["TotalPrice"].mean(),
    "Number of Products": df["Product"].nunique(),
    "Number of Payment Methods": df["PaymentMethod"].nunique(),
    "Number of Referral Sources": df["ReferralSource"].nunique(),
    "Number of Order Statuses": df["OrderStatus"].nunique()
}

summary_df = pd.DataFrame(summary.items(), columns=["Metric", "Value"])
summary_df

,Metric,Value
0,Total Orders,1.200000e+03
1,Total Customers,1.189000e+03
2,Total Revenue,1.264762e+06
3,Average Order Value,1.053968e+03
4,Number of Products,7.000000e+00
5,Number of Payment Methods,5.000000e+00
6,Number of Referral Sources,5.000000e+00
7,Number of Order Statuses,5.000000e+00


In [22]:
category_columns = ["Product", "PaymentMethod", "OrderStatus", "ReferralSource", "HasCoupon"]

for col in category_columns:
    print(f"\n{col} Distribution")
    print(df[col].value_counts().reset_index().rename(
        columns={col: "Category", "count": "Count"}
    ))


Product Distribution
  Category  Count
0  Printer    181
1   Tablet    179
2    Chair    178
3   Laptop    173
4     Desk    170
5  Monitor    163
6    Phone    156

PaymentMethod Distribution
      Category  Count
0       Online    258
1         Cash    246
2  Credit Card    234
3   Debit Card    232
4    Gift Card    230

OrderStatus Distribution
    Category  Count
0  Cancelled    250
1   Returned    247
2    Pending    237
3    Shipped    235
4  Delivered    231

ReferralSource Distribution
    Category  Count
0  Instagram    259
1      Email    250
2     Google    241
3   Facebook    228
4   Referral    222

HasCoupon Distribution
      Category  Count
0  Coupon Used    891
1    No Coupon    309


## Data Verification Summary

This notebook was created for DecodeLabs Data Analytics Project 2.

Project 1 focused mainly on checking whether the retail orders dataset was clean, reliable, and ready to use.  
Project 2 is different because it focuses on Exploratory Data Analysis (EDA), which means discovering patterns, trends, outliers, relationships, and business insights from the data.

Before starting the full EDA process, a quick verification was completed to make sure the dataset is suitable for analysis.

### Checks Completed

- Checked the number of rows and columns
- Reviewed all column names
- Checked data types
- Verified the Date column format
- Checked missing values
- Checked duplicate Order IDs
- Checked the date range of the dataset
- Verified that TotalPrice matches Quantity × UnitPrice
- Created a new HasCoupon column for coupon usage analysis
- Reviewed key category columns such as Product, PaymentMethod, OrderStatus, ReferralSource, and HasCoupon

### Coupon Code Note

The CouponCode column was already cleaned in Project 1, so some missing coupon values may have already been replaced with text such as "No Coupon".

Because of that, Project 2 does not only check for null or blank values.  
It also checks for text values that represent no coupon usage, such as "No Coupon", "None", "Null", "NA", and similar values.

This is important because coupon usage will be used later for EDA.  
For example, we can compare whether customers who used coupons spent more than customers who did not use coupons.

### Project 2 Dataset Decision

Since this dataset was already cleaned in Project 1, this notebook does not repeat the full cleaning process.

Instead, this notebook performs a light verification step and prepares the dataset for EDA.

The verified dataset will be used in the next phases for:

- Descriptive statistics
- Distribution analysis
- Outlier detection
- Trend analysis
- Correlation analysis
- Product analysis
- Payment method analysis
- Coupon usage analysis
- Referral source analysis
- Order status analysis
- Business insights and recommendations

### Conclusion

The dataset is suitable for Project 2 Exploratory Data Analysis.

Project 1 answered the question:

"Is the data clean and reliable?"

Project 2 will answer the question:

"What patterns, trends, outliers, and business insights can we discover from the data?"

In [23]:
# Remove temporary calculation column before saving
df = df.drop(columns=["CalculatedTotalPrice"], errors="ignore")

# Save verified dataset for Project 2 EDA
df.to_csv("../data/processed/ecommerce_orders_project2_verified.csv", index=False)

print("Project 2 verified dataset saved successfully.")

Project 2 verified dataset saved successfully.
